# PyTorch raw dataset and CUDA verification

This notebook validates the CPU-only `TorchDataset` adapter, variable-length collation, pinned host memory, non-blocking CUDA transfer, and a minimal finite `Conv1d` forward/backward pass.

It uses one canonical `Data_Train/exec` block and one canonical `Data_Pattern/patt` block. No prediction target or train/test interpretation is introduced.

In [1]:
from pathlib import Path
import sys

import torch
from torch.utils.data import ConcatDataset, DataLoader, Subset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.datasets import NumpyDataset, TorchDataset, collate_torch_samples

assert torch.cuda.is_available(), "This verification notebook requires CUDA"
DEVICE = torch.device("cuda")
{
    "torch_version": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "device": torch.cuda.get_device_name(DEVICE),
}

{'torch_version': '2.12.0+cu130',
 'cuda_runtime': '13.0',
 'device': 'NVIDIA GeForce RTX 3070 Ti'}

## Canonical source datasets

`NumpyDataset` remains responsible for FIF loading and caching. `TorchDataset` only creates CPU tensors that share the NumPy storage.

In [2]:
exec_dataset = TorchDataset(
    NumpyDataset(
        PROJECT_ROOT / "data" / "Data_Train",
        dataset_step_type="exec",
        cache_policy="disk",
    )
)
patt_dataset = TorchDataset(
    NumpyDataset(
        PROJECT_ROOT / "data" / "Data_Pattern",
        dataset_step_type="patt",
        cache_policy="disk",
    )
)

exec_sample = exec_dataset[0]
patt_sample = patt_dataset[0]
assert exec_sample.eeg.device.type == "cpu"
assert patt_sample.eeg.device.type == "cpu"
assert exec_sample.eeg.shape[0] == patt_sample.eeg.shape[0] == 63
{
    "Data_Train/exec": tuple(exec_sample.eeg.shape),
    "Data_Pattern/patt": tuple(patt_sample.eeg.shape),
    "dtype": str(exec_sample.eeg.dtype),
}

{'Data_Train/exec': (63, 16001),
 'Data_Pattern/patt': (63, 26001),
 'dtype': 'torch.float32'}

## Variable-length pinned batch

The batch contains both recording families to exercise real zero padding and masks. EOG NaNs remain unchanged and are represented by `eog_finite_mask`.

In [3]:
combined = ConcatDataset([Subset(exec_dataset, [0]), Subset(patt_dataset, [0])])
loader = DataLoader(
    combined,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_torch_samples,
)
batch = next(iter(loader))

assert batch.lengths.tolist() == [exec_sample.eeg.shape[-1], patt_sample.eeg.shape[-1]]
assert batch.eeg.shape == (2, 63, max(batch.lengths).item())
assert batch.time_mask.sum(dim=1).tolist() == batch.lengths.tolist()
assert torch.count_nonzero(batch.eeg[0, :, batch.lengths[0] :]) == 0
assert batch.eeg.is_pinned() and batch.eog.is_pinned()
assert batch.lengths.is_pinned() and batch.time_mask.is_pinned()
assert batch.eog_finite_mask.is_pinned()

for index, length in enumerate(batch.lengths.tolist()):
    assert torch.equal(
        batch.eog_finite_mask[index, :, :length],
        torch.isfinite(batch.eog[index, :, :length]),
    )
    assert not batch.eog_finite_mask[index, :, length:].any()

{
    "batch_shape": tuple(batch.eeg.shape),
    "lengths": batch.lengths.tolist(),
    "pinned": batch.eeg.is_pinned(),
    "finite_eog_fraction": float(batch.eog_finite_mask.float().mean()),
}

{'batch_shape': (2, 63, 26001),
 'lengths': [16001, 26001],
 'pinned': True,
 'finite_eog_fraction': 0.779170036315918}

## CUDA forward/backward smoke test

The model is deliberately tiny and untrained. This verifies device transfer and autograd only; it is not an EEG baseline or scientific result.

In [4]:
gpu_batch = batch.to(DEVICE, non_blocking=True)
assert gpu_batch.eeg.is_cuda
assert gpu_batch.eog.is_cuda
assert gpu_batch.lengths.is_cuda
assert gpu_batch.samples is batch.samples

model = torch.nn.Sequential(
    torch.nn.Conv1d(gpu_batch.eeg.shape[1], 8, kernel_size=7, stride=8),
    torch.nn.GELU(),
    torch.nn.AdaptiveAvgPool1d(1),
    torch.nn.Flatten(),
    torch.nn.Linear(8, 1),
).to(DEVICE)

prediction = model(gpu_batch.eeg)
loss = prediction.square().mean()
loss.backward()
torch.cuda.synchronize()

assert torch.isfinite(prediction).all()
assert torch.isfinite(loss)
assert all(parameter.grad is not None for parameter in model.parameters())
assert all(torch.isfinite(parameter.grad).all() for parameter in model.parameters())

result = {
    "status": "CUDA_VERIFIED",
    "device": str(DEVICE),
    "prediction_shape": tuple(prediction.shape),
    "loss": float(loss.detach().cpu()),
    "all_gradients_finite": True,
}
result

{'status': 'CUDA_VERIFIED',
 'device': 'cuda',
 'prediction_shape': (2, 1),
 'loss': 0.0016507472610101104,
 'all_gradients_finite': True}

## Conclusion

The raw adapter remains CPU-only, real variable-length blocks collate with explicit masks, pinned batches transfer to CUDA, and a minimal `Conv1d` forward/backward pass succeeds with finite outputs and gradients.